# Q1. Se eu reprovar em uma obrigatória, quanto isso me atrasa?

**Pergunta.** Ao reprovar em uma disciplina obrigatória, quantas outras
disciplinas do curso são adiadas por causa dos pré-requisitos, e a
formatura atrasa?

**Cursos analisados.** Os que têm alunos matriculados na turma de 2026/2
da SCC0252 (Visualização Computacional), segundo a tabela `vagas`: Ciências
de Computação, Ciência de Dados (para quem a disciplina é obrigatória),
Sistemas de Informação e Engenharia de Computação.

**Como medimos.** Para cada curso, simulamos o aluno que segue o semestre
ideal da grade. Ao reprovar, ele refaz a disciplina na próxima oferta, e
tudo o que depende dela (direta ou indiretamente) é adiado. Dois cenários
podem ser alternados na própria visualização:

- **oferta**: *anual* (a obrigatória só é oferecida no semestre de mesma
  paridade do seu semestre ideal) ou *semestral* (pode refazer no semestre
  seguinte);
- **reprovação**: *forte* (nota < 3 ou frequência < 70 %, bloqueia
  `requisito` e `requisito fraco`) ou *branda* (bloqueia só `requisito`).

Os **estágios supervisionados** são oferecidos todo semestre, então para
eles vale sempre a oferta semestral. `indicação de conjunto` não bloqueia
e não há limite de créditos por semestre.

In [1]:
# Só faz algo no Google Colab: clona o repositório, entra em notebooks/ e
# instala o que faltar. Rodando localmente, esta célula não faz nada.
import sys
if "google.colab" in sys.modules:
    import importlib.util, os, subprocess
    REPO = "https://github.com/vmoreli/viscomp-projeto"
    PASTA = "/content/viscomp-projeto"
    if not os.path.exists(PASTA):
        subprocess.run(["git", "clone", "--depth", "1", REPO, PASTA], check=True)
    os.chdir(os.path.join(PASTA, "notebooks"))
    faltando = [m for m in ("plotly", "networkx") if importlib.util.find_spec(m) is None]
    if faltando:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando], check=True)

In [2]:
from pathlib import Path
import pandas as pd
import networkx as nx
import plotly.graph_objects as go

DADOS = Path("../dados/processado")   # pasta extraída de dados.zip
if not DADOS.exists():
    DADOS = Path("dados/processado")

grade = pd.read_csv(DADOS / "grade_curricular.csv")
req = pd.read_csv(DADOS / "requisitos.csv")
cursos = pd.read_csv(DADOS / "cursos.csv")
disc = pd.read_csv(DADOS / "disciplinas.csv",
                   usecols=["sigla", "nome", "departamento"])

# codhab vem como float com NaN em requisitos
req["codhab"] = req["codhab"].fillna(0).astype(int)
grade["codhab"] = grade["codhab"].astype(int)
NOME = disc.set_index("sigla")["nome"].to_dict()

# Estágios supervisionados são oferecidos todo semestre, qualquer que seja o
# cenário de oferta das demais obrigatórias
OFERTA_SEMESTRAL = {s for s, n in NOME.items() if str(n).startswith("Estágio Supervisionado")}

cursos["rotulo"] = (cursos["nome_curso"]
                    .str.replace("Bacharelado em ", "Bach. ")
                    .str.replace("Licenciatura em ", "Lic. "))
# Núcleo Geral de Matemática tem só 5 obrigatórias e nenhum requisito
cursos = cursos[cursos["codhab"] != 100].reset_index(drop=True)


def montar_curso(codcur, codhab, bloqueia_fraco=True):
    # Retorna (ideal, grupos):
    #   ideal[sigla]  = semestre ideal da obrigatória
    #   grupos[sigla] = lista de conjuntos de requisitos (basta cumprir um)
    g = grade[(grade.codcur == codcur) & (grade.codhab == codhab)
              & (grade.tipo_vinculo == "obrigatoria")]
    ideal = (g.drop_duplicates("sigla").set_index("sigla")["semestre_ideal"]
              .astype(int).to_dict())

    tipos = {"requisito", "requisito fraco"} if bloqueia_fraco else {"requisito"}
    r = req[(req.codcur == codcur) & (req.codhab == codhab)
            & req.disciplina.isin(ideal) & req.tipo_requisito.isin(tipos)]

    grupos = {}
    for d, sub in r.groupby("disciplina"):
        gs = [set(x["requisito"]) for _, x in sub.groupby("grupo_alternativa")]
        # requisitos fora das obrigatórias (optativas, outros cursos) não
        # têm semestre conhecido; tiramos do grupo. Grupo vazio = livre.
        gs = [gr & ideal.keys() for gr in gs]
        if all(gs):
            grupos[d] = gs
    return ideal, grupos


def grafo(ideal, grupos):
    G = nx.DiGraph()
    G.add_nodes_from(ideal)
    for d, gs in grupos.items():
        for gr in gs:
            G.add_edges_from((p, d) for p in gr)
    return G


def proxima_oferta(t, ideal_d, oferta):
    # Primeiro semestre >= t em que a disciplina é oferecida
    if oferta == "semestral":
        return t
    return t if (t - ideal_d) % 2 == 0 else t + 1


def agendar(ideal, grupos, G, reprovada=None, oferta="anual"):
    # Semestre em que cada obrigatória é concluída.
    # A reprovada é cursada no seu semestre, reprovada, e refeita na
    # próxima oferta.
    s = {}
    for d in nx.topological_sort(G):
        t = ideal[d]
        if d in grupos:
            t = max(t, min(max(s[p] for p in gr) + 1 for gr in grupos[d]))
        of = "semestral" if d in OFERTA_SEMESTRAL else oferta
        s[d] = proxima_oferta(t, ideal[d], of)
        if d == reprovada:
            s[d] = proxima_oferta(s[d] + 1, ideal[d], of)
    return s


def analisar_curso(codcur, codhab, oferta="anual", bloqueia_fraco=True):
    # Uma linha por obrigatória: atraso na formatura se reprovar nela.
    ideal, grupos = montar_curso(codcur, codhab, bloqueia_fraco)
    G = grafo(ideal, grupos)
    base = agendar(ideal, grupos, G, oferta=oferta)
    T = max(base.values())
    linhas = []
    for d in ideal:
        s = agendar(ideal, grupos, G, reprovada=d, oferta=oferta)
        linhas.append({
            "sigla": d,
            "nome": NOME.get(d, d),
            "semestre_ideal": ideal[d],
            "semestre_base": base[d],
            "atraso": max(s.values()) - T,
            "empurradas": sum(s[x] > base[x] for x in ideal if x != d),
            "descendentes": len(nx.descendants(G, d)),
        })
    df = pd.DataFrame(linhas).sort_values(["semestre_ideal", "sigla"])
    df["inconsistente"] = df.semestre_base > df.semestre_ideal
    return df, ideal, grupos, G, base



def posicoes(nos, ideal, G, varreduras=4):
    # Layout em camadas (Sugiyama): x = semestre ideal; y = ordem na coluna
    # pela heurística do baricentro, para reduzir cruzamentos de arestas
    colunas = {}
    for d in sorted(nos):
        colunas.setdefault(ideal[d], []).append(d)
    sems = sorted(colunas)
    y = {d: i for s in sems for i, d in enumerate(colunas[s])}

    def reordena(s, vizinhos):
        def chave(d):
            v = [y[u] for u in vizinhos(d) if u in y]
            return sum(v) / len(v) if v else y[d]
        colunas[s].sort(key=chave)
        for i, d in enumerate(colunas[s]):
            y[d] = i

    for _ in range(varreduras):
        for s in sems[1:]:
            reordena(s, G.predecessors)
        for s in sems[-2::-1]:
            reordena(s, G.successors)
    return {d: (ideal[d], (len(colunas[ideal[d]]) - 1) / 2 - y[d]) for d in nos}


# cursos com alunos matriculados na turma atual de SCC0252 (tabela vagas).
# A tabela vagas nomeia o curso por extenso; o mapa liga esse nome ao codcur.
DISCIPLINA = "SCC0252"
CURSO_VAGAS = {"ICMC - Bacharelado em Ciência de Dados": (55090, 0),
               "ICMC - Ciências de Computação": (55041, 0),
               "ICMC - Bacharelado em Sistemas de Informação": (55051, 4),
               "EESC e ICMC - Engenharia de Computação": (97001, 0),
               "ICMC -  Bacharelado em Estatística e Ciência de Dados": (55071, 4),
               "ICMC - Bach Mat Aplic Computação Cient": (55060, 0)}
vagas = pd.read_csv(DADOS / "vagas.csv")
turma = vagas[(vagas.sigla == DISCIPLINA) & (vagas.curso != "TOTAL") & (vagas.matriculados > 0)]
ALUNOS = (turma.assign(chave=turma.curso.map(CURSO_VAGAS)).dropna(subset=["chave"])
          .groupby("chave").matriculados.sum().to_dict())
CENARIOS = [(o, f) for o in ("anual", "semestral") for f in (True, False)]


def dados_curso(codcur, codhab):
    # Tudo o que a visualização precisa de um curso, nos 4 cenários.
    # O layout vem do grafo completo (requisitos fortes e fracos), para os nós
    # não mudarem de lugar ao trocar de cenário.
    _, ideal, _, G, _ = analisar_curso(codcur, codhab, "anual", True)
    fortes = set(map(tuple, req[(req.codcur == codcur) & (req.codhab == codhab)
                               & (req.tipo_requisito == "requisito")]
                     [["requisito", "disciplina"]].values))
    cen, com_efeito = {}, set()
    for oferta, forte in CENARIOS:
        df, ideal, grupos, Gc, base = analisar_curso(codcur, codhab, oferta, forte)
        imp = (df[(df.empurradas > 0) | (df.atraso > 0)]
               .sort_values(["empurradas", "atraso", "semestre_ideal"],
                            ascending=[False, False, True]))
        cascata = {}
        for r in imp.itertuples():
            s = agendar(ideal, grupos, Gc, reprovada=r.sigla, oferta=oferta)
            cascata[r.sigla] = {d: int(s[d]) for d in ideal if s[d] > base[d] or d == r.sigla}
            com_efeito |= set(cascata[r.sigla])
        cen[f"{oferta}|{'forte' if forte else 'branda'}"] = dict(
            impacto=[dict(s=r.sigla, e=int(r.empurradas), a=int(r.atraso)) for r in imp.itertuples()],
            cascata=cascata, n=len(df), adiam=int((df.empurradas > 0).sum()),
            atrasam=int((df.atraso > 0).sum()))
    nos = sorted({d for d in G if G.degree(d) > 0} | com_efeito)
    pos = posicoes(nos, ideal, G)
    nome = cursos[(cursos.codcur == codcur) & (cursos.codhab == codhab)].rotulo.iloc[0]
    return dict(
        nome=nome, alunos=int(ALUNOS[(codcur, codhab)]), maxsem=max(ideal.values()),
        nos=[dict(s=d, x=pos[d][0], y=round(pos[d][1], 3)) for d in nos],
        arestas=[[p, q, (p, q) in fortes] for p, q in G.edges],
        nomes={d: NOME.get(d, d) for d in ideal}, sem={d: int(v) for d, v in ideal.items()},
        cen=cen)


ORDEM = sorted(ALUNOS, key=lambda k: -ALUNOS[k])   # curso com mais alunos primeiro
DADOS_VIS = {"ordem": [f"{a}-{b}" for a, b in ORDEM],
             "cursos": {f"{a}-{b}": dados_curso(a, b) for a, b in ORDEM}}
pd.DataFrame([{"curso": DADOS_VIS["cursos"][k]["nome"], f"alunos em {DISCIPLINA}": DADOS_VIS["cursos"][k]["alunos"]}
              for k in DADOS_VIS["ordem"]])

,curso,alunos em SCC0252
0,Bach. Ciências de Computação,22
1,Bach. Ciência de Dados,20
2,Bach. Sistemas de Informação,3
3,Engenharia de Computação,2


## Visualizações coordenadas

Duas visões ligadas, sempre do mesmo curso (um aluno pertence a um curso
só, então o curso é escolhido no menu, não comparado lado a lado):

1. **Quais reprovações custam mais?** Gráfico de **barras ordenado**:
   disciplinas adiadas (quantitativo) por disciplina reprovada
   (categórico). Azul (barra e texto "formatura +k") indica que a formatura
   também atrasa. O resumo acima das visões lista as siglas dessas
   disciplinas, inclusive as que não adiam nenhuma outra (barra de
   comprimento 0, como o último estágio).
2. **Por onde a cascata se propaga?** **Diagrama de nós e arestas em
   camadas**, o mais indicado para analisar caminhos: x = semestre ideal,
   ordem na coluna escolhida para reduzir cruzamentos. O losango vermelho é
   a reprovada, os círculos laranja são as adiadas; contínua é `requisito`, pontilhada é
   `requisito fraco`.

**Interação**, seguindo o mantra *overview first, zoom and filter, then
details on demand*:

- **Menu de curso** e **botões de cenário** (oferta anual/semestral,
  reprovação forte/branda): consultas dinâmicas, e as duas visões e o
  resumo se atualizam na hora;
- **Clique numa barra** (visão geral) → a visão 2 mostra a cascata daquela
  disciplina (detalhe). É o *linking* entre as visões coordenadas;
- **Passe o mouse num nó** da visão 2 → a barra correspondente fica em
  foco na visão 1 e as demais esmaecem (diferente da seleção, que é o
  contorno grosso com ▶ no rótulo); **clique no nó** para selecioná-lo
  direto no grafo;
- **Tooltips** (detalhes sob demanda): nas barras, nome completo e
  efeito da reprovação; nos nós, o deslocamento "semestre ideal →
  semestre novo" de cada disciplina adiada.

In [3]:
JS_PAINEL = r"""
const COR = {tinta: "#0b0b0b", tinta2: "#52514e", muda: "#898781", grade: "#e1e0d9",
             cinza: "#c3c2b7", barra: "#a9a8a0", cinzaClaro: "#e6e5df", azul: "#2a78d6",
             vermelho: "#d03b3b", laranja: "#ec835a", fundo: "#fcfcfb"};
const st = {curso: D.ordem[0], oferta: "anual", forte: "forte", sel: null};
const $ = (s) => root.querySelector(s);
const v1 = $(".v1"), v2 = $(".v2");
const cfg = {displayModeBar: false, responsive: true};
const fonte = {family: "Inter, Arial, sans-serif", size: 12, color: COR.tinta2};

const curso = () => D.cursos[st.curso];
const cenario = () => curso().cen[st.oferta + "|" + st.forte];
const rotulo = (s) => `${s} · ${curso().nomes[s].slice(0, 28)} (${curso().sem[s]}º)`;
let ordemV1 = [];   // siglas na ordem em que as barras foram desenhadas

// ---------- controles (consultas dinâmicas) ----------
const sel = $(".curso");
D.ordem.forEach((k) => {
  const o = document.createElement("option");
  o.value = k;
  o.textContent = D.cursos[k].nome;
  sel.appendChild(o);
});
sel.addEventListener("change", () => { st.curso = sel.value; st.sel = null; tudo(); });
root.querySelectorAll(".grp").forEach((g) => {
  g.querySelectorAll("button").forEach((b) => b.addEventListener("click", () => {
    st[g.dataset.k] = b.dataset.v;
    tudo();
  }));
});
function marcaBotoes() {
  root.querySelectorAll(".grp").forEach((g) => g.querySelectorAll("button").forEach((b) =>
    b.classList.toggle("on", st[g.dataset.k] === b.dataset.v)));
}

// ---------- visão 1: barras ordenadas ----------
// seleção (clique): contorno grosso + rótulo em negrito com ▶
// hover vindo da visão 2: a barra correspondente fica em foco e as outras esmaecem
function larguras() {
  return ordemV1.map((s) => (s === st.sel ? 2.5 : 0));
}
function opacidades(destaque) {
  return ordemV1.map((s) => (!destaque || s === destaque ? 1 : 0.3));
}
function rotulosEixo() {
  return ordemV1.map((s) => (s === st.sel ? `<b>▶ ${rotulo(s)}</b>` : rotulo(s)));
}
function desenhaV1() {
  const imp = cenario().impacto;
  if (!imp.some((r) => r.s === st.sel)) st.sel = imp.length ? imp[0].s : null;
  const rev = imp.slice().reverse();   // plotly desenha de baixo para cima
  ordemV1 = rev.map((r) => r.s);
  const y = rev.map((r) => rotulo(r.s));
  const xmax = Math.max(4, ...imp.map((r) => r.e));
  const dica = rev.map((r) => [r.s, curso().nomes[r.s], r.e, r.a]);
  const alvo = {   // faixa invisível: torna a linha toda clicável, mesmo com barra curta
    type: "bar", orientation: "h", y, x: rev.map(() => xmax * 1.6), customdata: dica,
    marker: {color: "rgba(0,0,0,0)"}, hovertemplate:
      "<b>%{customdata[0]}</b> %{customdata[1]}<br>adia %{customdata[2]} disciplina(s)" +
      "<br>formatura +%{customdata[3]} semestre(s)<br><i>clique para ver a cascata</i><extra></extra>"};
  const barras = {
    type: "bar", orientation: "h", y, x: rev.map((r) => r.e), customdata: dica,
    hoverinfo: "skip", cliponaxis: false, textposition: "outside",
    text: rev.map((r) => `${r.e}` +
      (r.a ? `   formatura +${r.a}` : "")),
    // quem atrasa a formatura ganha o texto em azul, além da barra azul
    textfont: {size: 11, color: rev.map((r) => (r.a > 0 ? COR.azul : COR.tinta2))},
    marker: {color: rev.map((r) => (r.a > 0 ? COR.azul : COR.barra)),
             opacity: opacidades(), line: {color: COR.tinta, width: larguras()}}};
  const layout = {
    font: fonte, paper_bgcolor: COR.fundo, plot_bgcolor: COR.fundo, barmode: "overlay",
    bargap: 0.28, height: Math.max(200, 70 + 22 * rev.length), showlegend: false,
    margin: {l: 10, r: 10, t: 10, b: 45},
    xaxis: {title: {text: "disciplinas adiadas"}, range: [0, xmax * 1.6], gridcolor: COR.grade,
            zeroline: false, fixedrange: true},
    yaxis: {automargin: true, tickfont: {size: 11}, fixedrange: true,
            tickmode: "array", tickvals: y, ticktext: rotulosEixo()},
    annotations: imp.length ? [] : [{text: "Nenhuma reprovação adia outra disciplina neste cenário",
      xref: "paper", yref: "paper", x: 0.5, y: 0.5, showarrow: false, font: {size: 13}}]};
  return Plotly.react(v1, [alvo, barras], layout, cfg);
}

// ---------- visão 2: nós e arestas em camadas ----------
function desenhaV2() {
  const c = curso(), cen = cenario();
  const casc = st.sel ? cen.cascata[st.sel] : {};
  const naCasc = (s) => s in casc;
  const pos = Object.fromEntries(c.nos.map((n) => [n.s, n]));
  const tracos = [];
  // arestas: contexto (cinza) e cascata (escuras); contínua = requisito, pontilhada = fraco
  for (const destaque of [false, true]) {
    for (const forte of [true, false]) {
      const xs = [], ys = [];
      c.arestas.forEach(([p, q, f]) => {
        if (f !== forte || !(p in pos) || !(q in pos)) return;
        // na reprovação branda, requisito fraco não propaga a cascata
        const ehCasc = naCasc(p) && naCasc(q) && (f || st.forte === "forte");
        if (ehCasc !== destaque) return;
        xs.push(pos[p].x, pos[q].x, null); ys.push(pos[p].y, pos[q].y, null);
      });
      tracos.push({type: "scatter", mode: "lines", x: xs, y: ys, hoverinfo: "skip",
        line: {color: destaque ? COR.tinta2 : COR.grade, width: destaque ? 2 : 1,
               dash: forte ? "solid" : "dot"}});
    }
  }
  // nós em três grupos
  const grupos = [
    {lst: c.nos.filter((n) => !naCasc(n.s)), cor: COR.cinzaClaro, tam: 9, txt: (n) => n.s,
     tf: {size: 9, color: COR.muda}},
    {lst: c.nos.filter((n) => naCasc(n.s) && n.s !== st.sel), cor: COR.laranja, tam: 12,
     txt: (n) => `<b>${n.s}</b>`, tf: {size: 11, color: COR.tinta}},
    {lst: c.nos.filter((n) => n.s === st.sel), cor: COR.vermelho, tam: 17, forma: "diamond",
     txt: (n) => `<b>${n.s}</b>`, tf: {size: 11, color: COR.tinta}}];
  grupos.forEach((g) => tracos.push({
    type: "scatter", mode: "markers+text", x: g.lst.map((n) => n.x), y: g.lst.map((n) => n.y),
    // rótulos alternados (semestre ímpar em cima, par embaixo) para não colidirem
    text: g.lst.map(g.txt), textposition: g.lst.map((n) => (n.x % 2 ? "top center" : "bottom center")),
    textfont: g.tf, customdata: g.lst.map((n) => n.s),
    marker: {size: g.tam, color: g.cor, symbol: g.forma || "circle", line: {color: g.cor === COR.cinzaClaro ? COR.cinza : COR.fundo, width: 1}},
    hovertext: g.lst.map((n) => `<b>${n.s}</b> ${c.nomes[n.s]}<br>` + (
      n.s === st.sel ? `reprovada no ${c.sem[n.s]}º → refaz no ${casc[n.s]}º`
      : naCasc(n.s) ? `adiada: ${c.sem[n.s]}º → ${casc[n.s]}º`
      : `semestre ideal: ${c.sem[n.s]}º`)), hoverinfo: "text"}));

  const ys = c.nos.map((n) => n.y);
  const cols = {};
  c.nos.forEach((n) => { cols[n.x] = (cols[n.x] || 0) + 1; });
  const layout = {
    font: fonte, paper_bgcolor: COR.fundo, plot_bgcolor: COR.fundo, showlegend: false,
    height: 130 + 62 * Math.max(...Object.values(cols)), margin: {l: 10, r: 10, t: 10, b: 45},
    xaxis: {title: {text: "semestre ideal"}, dtick: 1, range: [0.2, c.maxsem + 1.0],
            gridcolor: COR.grade, zeroline: false},
    yaxis: {visible: false, range: [Math.min(...ys) - 0.8, Math.max(...ys) + 0.8]},
    hovermode: "closest"};
  // títulos fora do gráfico (HTML)
  const r = cen.impacto.find((x) => x.s === st.sel);
  $(".t2").innerHTML = st.sel
    ? `2. Reprovar em ${st.sel} (${c.nomes[st.sel]})`
    : "2. Por onde a cascata se propaga?";
  $(".sub").textContent = r
    ? `adia ${r.e} disciplina(s); formatura +${r.a} semestre(s)`
    : "nenhuma reprovação com efeito neste cenário";
  return Plotly.react(v2, tracos, layout, cfg);
}

function desenhaResumo() {
  const c = curso(), cen = cenario();
  $(".resumo").innerHTML = `<b>${c.nome}</b>: ${cen.n} obrigatórias · ` +
    `${cen.adiam} adiam outra disciplina se o aluno reprovar · ${cen.atrasam} atrasam a formatura` +
    (cen.atrasam ? ` <span style="color:${COR.azul}">(${cen.impacto.filter((r) => r.a > 0)
      .map((r) => r.s).join(", ")})</span>` : "");
}

function tudo() {
  marcaBotoes();
  desenhaResumo();
  return desenhaV1().then(desenhaV2);
}

// ---------- ligação entre as visões ----------
function seleciona(s) {
  if (!cenario().cascata[s]) return;
  st.sel = s;
  Plotly.restyle(v1, {"marker.line.width": [larguras()]}, [1]);
  Plotly.relayout(v1, {"yaxis.ticktext": rotulosEixo()});
  desenhaV2();
}
tudo().then(() => {
  v1.on("plotly_click", (ev) => seleciona(ev.points[0].customdata[0]));
  v2.on("plotly_click", (ev) => { const s = ev.points[0].customdata; if (s) seleciona(s); });
  v2.on("plotly_hover", (ev) => {
    const s = ev.points[0].customdata;
    if (s && ordemV1.includes(s)) Plotly.restyle(v1, {"marker.opacity": [opacidades(s)]}, [1]);
  });
  v2.on("plotly_unhover", () => Plotly.restyle(v1, {"marker.opacity": [opacidades()]}, [1]));
});

"""

import json, uuid
from IPython.display import HTML, display

EMBUTIR_PLOTLY = False   # True: embute o plotly.js no HTML (funciona sem internet, ~4 MB)

if EMBUTIR_PLOTLY:
    from plotly.offline import get_plotlyjs
    tag_plotly = f"<script>{get_plotlyjs()}</script>"
else:
    tag_plotly = '<script src="https://cdn.jsdelivr.net/npm/plotly.js-dist-min@2.35.2/plotly.min.js"></script>'

ID = "q1_" + uuid.uuid4().hex[:8]
HTML_PAINEL = f"""
<div id="{ID}" class="q1">
<style>
  #{ID} {{ font-family: Inter, Arial, sans-serif; color: #52514e; background: #fcfcfb;
          padding: 12px 8px; }}
  #{ID} .ctl {{ display: flex; flex-wrap: wrap; gap: 10px 18px; align-items: center; }}
  #{ID} select {{ font: inherit; padding: 4px 6px; border: 1px solid #c3c2b7;
                 border-radius: 4px; background: white; color: #0b0b0b; }}
  #{ID} .grp {{ display: inline-flex; border: 1px solid #c3c2b7; border-radius: 4px; overflow: hidden; }}
  #{ID} .grp button {{ font: inherit; font-size: 13px; padding: 4px 10px; border: 0;
                      background: white; color: #52514e; cursor: pointer; }}
  #{ID} .grp button + button {{ border-left: 1px solid #c3c2b7; }}
  #{ID} .grp button.on {{ background: #2a78d6; color: white; }}
  #{ID} .resumo {{ margin: 10px 0 4px; font-size: 14px; color: #0b0b0b; }}
  #{ID} .paineis {{ display: flex; flex-wrap: wrap; gap: 12px; }}
  #{ID} .p1 {{ flex: 1 1 380px; min-width: 320px; }}
  #{ID} .p2 {{ flex: 1.5 1 520px; min-width: 360px; }}
  #{ID} h4 {{ margin: 8px 0 0; font-size: 14px; color: #0b0b0b; font-weight: 600; }}
  #{ID} h4 small {{ font-weight: 400; color: #898781; }}
  #{ID} .sub {{ font-size: 12px; color: #52514e; min-height: 16px; }}
  #{ID} .leg {{ display: flex; flex-wrap: wrap; gap: 4px 14px; font-size: 12px; margin-top: 2px; }}
  #{ID} .leg i {{ display: inline-block; vertical-align: middle; margin-right: 5px; }}
  #{ID} .v1 .bars .point path {{ cursor: pointer; }}
</style>
<div class="ctl">
  <label>Curso <select class="curso"></select></label>
  <span class="grp" data-k="oferta"><button data-v="anual">Oferta anual</button><button data-v="semestral">Oferta semestral</button></span>
  <span class="grp" data-k="forte"><button data-v="forte">Reprovação forte</button><button data-v="branda">Reprovação branda</button></span>
</div>
<div class="resumo"></div>
<div class="paineis">
  <div class="p1"><h4>1. Quais reprovações custam mais? <small>clique numa barra</small></h4>
    <div class="leg"><span><i style="width:12px;height:10px;background:#a9a8a0"></i>formatura não atrasa</span>
      <span><i style="width:12px;height:10px;background:#2a78d6"></i>formatura atrasa</span></div>
    <div class="v1"></div></div>
  <div class="p2"><h4 class="t2"></h4><div class="sub"></div>
    <div class="leg"><span><i style="width:9px;height:9px;transform:rotate(45deg);margin:0 7px 0 2px;background:#d03b3b"></i>reprovada</span>
      <span><i style="width:9px;height:9px;border-radius:50%;background:#ec835a"></i>adiada (passe o mouse para ver o novo semestre)</span>
      <span><i style="width:7px;height:7px;border-radius:50%;background:#e6e5df;border:1px solid #c3c2b7"></i>sem efeito</span>
      <span><i style="width:18px;border-top:2px solid #52514e"></i>requisito</span>
      <span><i style="width:18px;border-top:2px dotted #52514e"></i>requisito fraco</span></div>
    <div class="v2"></div></div>
</div>
</div>
{tag_plotly}
<script>
(function() {{
const D = {json.dumps(DADOS_VIS, ensure_ascii=False)};
const root = document.getElementById("{ID}");
__JS__
}})();
</script>
""".replace("__JS__", JS_PAINEL)

with open("q1_interativo.html", "w", encoding="utf-8") as f:   # cópia para abrir no navegador
    f.write("<!doctype html><meta charset='utf-8'><title>Q1</title>" + HTML_PAINEL)
display(HTML(HTML_PAINEL))

## Leitura

*Cenário padrão da visualização: oferta anual e reprovação forte. Para
reproduzir cada ponto, escolha o curso no menu e clique na disciplina
indicada.*

**Resposta curta.** Nos quatro cursos da turma, uma reprovação em
obrigatória adia de 8 a 21 outras disciplinas no pior caso, sobretudo se
for no 1º ano. Mas quase nunca atrasa a formatura. O atraso na formatura
vem das disciplinas do fim da grade e, em Engenharia de Computação, da
cadeia de Circuitos.

**1. Reprovar no 1º ano custa mais.** Em todos os cursos, o topo da
visão 1 é de disciplinas do 1º e do 2º semestre: Introdução à Ciência de
Computação, AED I e Cálculo I. Em média, uma reprovação no 1º ou 2º
semestre adia 4,3 disciplinas em Sistemas de Informação, 2,6 em Ciência de
Dados e 2,5 em BCC, contra 0,1 a 0,3 a partir do 5º semestre. O extremo é
Sistemas de Informação: reprovar em SSC0501 (Introdução à Ciência de
Computação I) adia 21 disciplinas, e em SCC0502 (AED I) adia 17. Clicando
nessas barras, a visão 2 mostra a cascata partindo do 1º ou 2º semestre e
se abrindo em leque pelos semestres seguintes.

**2. A formatura quase não atrasa, porque o fim das grades tem folga.**
Na cascata da barra do topo de cada curso (selecionada por padrão),
passando o mouse nos nós laranja, os novos semestres param antes do
último. Em BCC, reprovar em SCC0202 (AED I) adia 13 disciplinas em 2
semestres cada, e as últimas (SSC0900 e SSC0904) vão para o 9º. Mesmo
assim, a formatura não atrasa: o 8º semestre não tem obrigatórias e o 9º e
o 10º só têm os estágios. O custo real é a sobrecarga dos semestres
seguintes, já que o modelo não limita créditos.

Na visão 1, as barras azuis (formatura atrasa) são raras em BCC, Sistemas
de Informação e Ciência de Dados: só 2 a 5 obrigatórias, todas do 6º
semestre em diante. Entre elas está o Estágio Supervisionado I: reprovar
nele atrasa a formatura em 1 semestre. Mesmo com os estágios oferecidos
todo semestre, o II depende do I e os dois ficam no fim do curso, sem
folga. Já a própria SCC0252, obrigatória do 6º semestre em Ciência de
Dados, nem aparece na visão 1: reprovar nela não adia nada nem atrasa a
formatura, porque ela não é requisito de outra obrigatória e pode ser
refeita no 8º.

**3. A exceção é Engenharia de Computação.** É o único curso com barras
azuis no topo da visão 1: 11 obrigatórias atrasam a formatura. Na visão 2,
reprovar em SEL0637 (Circuitos Elétricos Lineares, 2º semestre) desce por
uma cadeia de disciplinas de Circuitos da Elétrica até SEL0621, que vai
do 8º para o 10º semestre. Como essa cadeia vai até o fim do curso, não há
folga para absorver o atraso.

**4. A nota da reprovação decide se a cascata existe.** Na visão 2, as
arestas de BCC, Sistemas de Informação e Engenharia de Computação são
quase todas pontilhadas (`requisito fraco`). Com o botão **Reprovação
branda** (nota ≥ 3 e frequência ≥ 70 %), as cascatas quase somem: em
Sistemas de Informação nenhuma reprovação adia outra disciplina, em BCC e
Engenharia de Computação só uma, e em Ciência de Dados, que tem alguns
requisitos fortes (linhas contínuas), só duas.

**Sensibilidade.** Com o botão **Oferta semestral** (refazer no semestre
seguinte), as maiores cascatas diminuem: Sistemas de Informação vai de 21
para 13, BCC de 13 para 7, Engenharia de Computação de 9 para 5 e Ciência
de Dados de 8 para 6. O atraso máximo na formatura cai para 1 semestre.